# Random Tree Diabetes

## - Carga y ajuste de datos

Se elige empezar la prueba con XGboost

In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score

In [2]:
df = pd.read_csv('/workspaces/IgnacioSabinoG-IntroML/data/raw/diabetes.csv')

# Columnas donde el 0 es físicamente imposible (faltantes)
cols_con_ceros = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

# Convertimos a NaN para que el Boosting decida qué hacer con ellos
df[cols_con_ceros] = df[cols_con_ceros].replace(0, np.nan)

In [3]:
X = df.drop('Outcome', axis=1)
y = df['Outcome']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
#Configuración de búsqueda para Boosting
#'scale_pos_weight' es el equivalente a 'class_weight' en XGBoost
param_grid_xgb = {
    'n_estimators': [50, 100, 150],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.01, 0.1, 0.2], # Velocidad de aprendizaje (clave en Boosting)
    'subsample': [0.8, 1.0],           # % de datos usados por cada árbol
    'scale_pos_weight': [1, 2.3]       # 2.3 es aprox. el ratio de desbalanceo (102/43)
}

# 4. Grid Search
grid_xgb = GridSearchCV(
    XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    param_grid_xgb,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_xgb.fit(X_train, y_train)

# 5. Evaluación
best_xgb = grid_xgb.best_estimator_
y_pred_xgb = best_xgb.predict(X_test)

print(f"Mejores parámetros XGB: {grid_xgb.best_params_}")
print(f"Precisión XGB: {accuracy_score(y_test, y_pred_xgb):.4f}")
print("\nInforme de Clasificación XGBoost:")
print(classification_report(y_test, y_pred_xgb))

/home/vscode/.local/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [16:37:25] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/vscode/.local/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [16:37:25] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/vscode/.local/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [16:37:25] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/vscode/.local/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [16:37:25] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/vscode/.local/lib/

Mejores parámetros XGB: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 50, 'scale_pos_weight': 1, 'subsample': 0.8}
Precisión XGB: 0.7468

Informe de Clasificación XGBoost:
              precision    recall  f1-score   support

           0       0.82      0.78      0.80        99
           1       0.63      0.69      0.66        55

    accuracy                           0.75       154
   macro avg       0.73      0.73      0.73       154
weighted avg       0.75      0.75      0.75       154



/home/vscode/.local/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [16:37:50] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Respecto a los resultados, XGBoost (0.75) se ha quedado por debajo de tu Árbol de Decisión (0.83).

Esto puede pasar en este dataset específico por dos razones:

Ruido en los datos: Al no limpiar los NaN e incluir Insulin y SkinThickness, le estás dando al modelo variables que tienen muchísimos datos faltantes. El Boosting intenta encontrar patrones en ese "ruido" y a veces se confunde más que un árbol simple.

Sobreajuste (Overfitting): El Boosting es como un "fórmula 1"; si el circuito (el dataset) es pequeño y tiene baches (datos nulos), un coche más sencillo (un árbol de decisión bien podado) suele ganar la carrera.

### Conclusión
El Recall de la clase 1 (0.69) es mejor que el del primer Random Forest, pero peor que el del Árbol de Decisión equilibrado. Esto confirma que para este problema médico, la estructura jerárquica de un solo árbol es extremadamente potente.

¡Hola Ignacio! Está genial pero te faltan cosas propias del ejercicio: 1-No guardas modelo. 2- No incluyes comparación de los 3 modelos de los 3 proyectos. 3 - Solo usas XGBoost (pordrías probar grandientboosting, AdaBoost...)

## Prueba con el df limpio

In [5]:
df = pd.read_csv('/workspaces/IgnacioSabinoG-IntroML/data/raw/diabetes.csv')

df.drop(['Insulin', 'SkinThickness'], axis=1, inplace=True)

indices_a_eliminar = df[(df['Glucose'] == 0) | (df['BMI'] == 0) | (df['BloodPressure'] == 0)].index
df.drop(indices_a_eliminar, inplace=True)
df = df.reset_index(drop=True)
df

,Pregnancies,Glucose,BloodPressure,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,33.6,0.627,50,1
1,1,85,66,26.6,0.351,31,0
2,8,183,64,23.3,0.672,32,1
3,1,89,66,28.1,0.167,21,0
4,0,137,40,43.1,2.288,33,1
...,...,...,...,...,...,...,...
719,10,101,76,32.9,0.171,63,0
720,2,122,70,36.8,0.340,27,0
721,5,121,72,26.2,0.245,30,0
722,1,126,60,30.1,0.349,47,1


In [6]:
X = df.drop('Outcome', axis=1)
y = df['Outcome']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
param_grid_xgb = {
    'n_estimators': [50, 100, 150],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.01, 0.1, 0.2], # Velocidad de aprendizaje (clave en Boosting)
    'subsample': [0.8, 1.0],           # % de datos usados por cada árbol
    'scale_pos_weight': [1, 2.3]       # 2.3 es aprox. el ratio de desbalanceo (102/43)
}

# 4. Grid Search
grid_xgb = GridSearchCV(
    XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    param_grid_xgb,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_xgb.fit(X_train, y_train)

# 5. Evaluación
best_xgb = grid_xgb.best_estimator_
y_pred_xgb = best_xgb.predict(X_test)

print(f"Mejores parámetros XGB: {grid_xgb.best_params_}")
print(f"Precisión XGB: {accuracy_score(y_test, y_pred_xgb):.4f}")
print("\nInforme de Clasificación XGBoost:")
print(classification_report(y_test, y_pred_xgb))

/home/vscode/.local/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [16:37:51] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/vscode/.local/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [16:37:51] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/vscode/.local/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [16:37:51] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/vscode/.local/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [16:37:51] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


/home/vscode/.local/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [16:37:51] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/vscode/.local/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [16:37:51] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/vscode/.local/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [16:37:51] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/vscode/.local/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [16:37:51] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/vscode/.local/lib/

Mejores parámetros XGB: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 50, 'scale_pos_weight': 1, 'subsample': 1.0}
Precisión XGB: 0.7931

Informe de Clasificación XGBoost:
              precision    recall  f1-score   support

           0       0.86      0.84      0.85       102
           1       0.64      0.67      0.66        43

    accuracy                           0.79       145
   macro avg       0.75      0.76      0.76       145
weighted avg       0.80      0.79      0.79       145



/home/vscode/.local/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [16:38:08] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/vscode/.local/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [16:38:08] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/vscode/.local/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [16:38:08] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Mejoró sensiblemente confirmando que eliminar esas columnas es una buena idea en general.

XGBoost (Limpio): 0.79 Accuracy / 0.67 Recall Clase 1.

Árbol de Decisión (Limpio): 0.83 Accuracy / 0.72 Recall Clase 1.

### Conclusión
En este dataset de Diabetes, el Árbol de Decisión simple sigue siendo el rey.

Relaciones Lineales/Simples: Las reglas para detectar diabetes (Glucosa alta + IMC alto) son muy directas. Un solo árbol las encuentra rápido.

Tamaño de muestra: Tienes pocos datos (unos 700 registros). El Boosting (XGBoost) necesita miles de datos para "aprender" de sus errores y superar a un árbol básico.

Poder de la poda: Tu max_depth=4 en el árbol fue un ajuste perfecto que evitó el sobreajuste que los modelos más complejos están sufriendo.